In [7]:
import pandas as pd

# Download Premier League season 2023/24
url = "https://www.football-data.co.uk/mmz4281/2324/E0.csv"
df = pd.read_csv(url)

# First look
print(df.shape)
print(df.columns.tolist())

(380, 106)
['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH', 'IWD', 'IWA', 'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', 'VCH', 'VCD', 'VCA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'AHh', 'B365AHH', 'B365AHA', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'B365CH', 'B365CD', 'B365CA', 'BWCH', 'BWCD', 'BWCA', 'IWCH', 'IWCD', 'IWCA', 'PSCH', 'PSCD', 'PSCA', 'WHCH', 'WHCD', 'WHCA', 'VCCH', 'VCCD', 'VCCA', 'MaxCH', 'MaxCD', 'MaxCA', 'AvgCH', 'AvgCD', 'AvgCA', 'B365C>2.5', 'B365C<2.5', 'PC>2.5', 'PC<2.5', 'MaxC>2.5', 'MaxC<2.5', 'AvgC>2.5', 'AvgC<2.5', 'AHCh', 'B365CAHH', 'B365CAHA', 'PCAHH', 'PCAHA', 'MaxCAHH', 'MaxCAHA', 'AvgCAHH', 'AvgCAHA']


In [8]:
# Columns we actually need
USEFUL_COLS = [
    # Match info
    'Date', 'HomeTeam', 'AwayTeam', 'Referee',
    
    # Full time result
    'FTHG',   # Full Time Home Goals
    'FTAG',   # Full Time Away Goals
    'FTR',    # Full Time Result (H/D/A)
    
    # Half time (useful feature later)
    'HTHG',   # Half Time Home Goals
    'HTAG',   # Half Time Away Goals
    
    # Match stats
    'HS',     # Home Shots
    'AS',     # Away Shots
    'HST',    # Home Shots on Target
    'AST',    # Away Shots on Target
    'HC',     # Home Corners
    'AC',     # Away Corners
    'HF',     # Home Fouls
    'AF',     # Away Fouls
    'HY',     # Home Yellow Cards
    'AY',     # Away Yellow Cards
    'HR',     # Home Red Cards
    'AR',     # Away Red Cards
]

df = df[USEFUL_COLS].copy()

# Fix date format
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')

# Sort by date (important for time series later)
df = df.sort_values('Date').reset_index(drop=True)

print(df.shape)
df.head()

(380, 21)


,Date,HomeTeam,AwayTeam,Referee,FTHG,FTAG,FTR,HTHG,HTAG,HS,...,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR
0,2023-08-11,Burnley,Man City,C Pawson,0,3,A,0,2,6,...,1,8,6,5,11,8,0,0,1,0
1,2023-08-12,Arsenal,Nott'm Forest,M Oliver,2,1,H,2,0,15,...,7,2,8,3,12,12,2,2,0,0
2,2023-08-12,Bournemouth,West Ham,P Bankes,1,1,D,0,0,14,...,5,3,10,4,9,14,1,4,0,0
3,2023-08-12,Brighton,Luton,D Coote,4,1,H,1,0,27,...,12,3,6,7,11,12,2,2,0,0
4,2023-08-12,Everton,Fulham,S Attwell,0,1,A,0,0,19,...,9,2,10,4,12,6,0,2,0,0


In [12]:
# Check for nulls
print(df.isnull().sum())
print(f"\nTotal matches: {len(df)}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\nUnique referees: {df['Referee'].nunique()}")
print(f"Unique teams: {df['HomeTeam'].nunique()}")

Date        1
HomeTeam    1
AwayTeam    1
FTHG        1
FTAG        1
FTR         1
HTHG        1
HTAG        1
Referee     1
HS          1
AS          1
HST         1
AST         1
HF          1
AF          1
HC          1
AC          1
HY          1
AY          1
HR          1
AR          1
Season      0
dtype: int64

Total matches: 4530
Date range: 2014-08-16 00:00:00 to 2026-05-04 00:00:00

Unique referees: 53
Unique teams: 35


In [29]:
# Load multiple Premier League seasons
SEASONS = {
    '2526': '2025/26',
    '2425': '2024/25',
    '2324': '2023/24',
    '2223': '2022/23',
    '2122': '2021/22',
    '2021': '2020/21',
    '1920': '2019/20',
    '1819': '2018/19',
    '1718': '2017/18',
    '1617': '2016/17',
    '1516': '2015/16',
    '1415': '2014/15',
}

dfs = []

for code, name in SEASONS.items():
    url = f"https://www.football-data.co.uk/mmz4281/{code}/E0.csv"
    try:
        temp = pd.read_csv(url, usecols=lambda c: c in USEFUL_COLS)
        temp['Season'] = name
        dfs.append(temp)
        print(f"✅ {name} → {len(temp)} matches loaded")
    except Exception as e:
        print(f"❌ {name} → {e}")

# Combine all seasons
df = pd.concat(dfs, ignore_index=True)
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)
df = df.sort_values('Date').reset_index(drop=True)

# Basic info
print(f"\nTotal matches: {len(df)}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Unique referees: {df['Referee'].nunique()}")
print(f"Unique teams: {df['HomeTeam'].nunique()}")


✅ 2025/26 → 349 matches loaded
✅ 2024/25 → 380 matches loaded
✅ 2023/24 → 380 matches loaded
✅ 2022/23 → 380 matches loaded
✅ 2021/22 → 380 matches loaded
✅ 2020/21 → 380 matches loaded
✅ 2019/20 → 380 matches loaded
✅ 2018/19 → 380 matches loaded
✅ 2017/18 → 380 matches loaded
✅ 2016/17 → 380 matches loaded
✅ 2015/16 → 380 matches loaded
✅ 2014/15 → 381 matches loaded

Total matches: 4530
Date range: 2014-08-16 00:00:00 to 2026-05-04 00:00:00
Unique referees: 53
Unique teams: 35


In [30]:
# Null analysis
print("\nNULL VALUES:")
print(df.isnull().sum().sort_values(ascending=False))

# Null percentage
print("\nNULL %:")
print((df.isnull().mean() * 100).sort_values(ascending=False))


NULL VALUES:
Date        1
HomeTeam    1
AR          1
HR          1
AY          1
HY          1
AC          1
HC          1
AF          1
HF          1
AST         1
HST         1
AS          1
HS          1
Referee     1
HTAG        1
HTHG        1
FTR         1
FTAG        1
FTHG        1
AwayTeam    1
Season      0
dtype: int64

NULL %:
Date        0.022075
HomeTeam    0.022075
AR          0.022075
HR          0.022075
AY          0.022075
HY          0.022075
AC          0.022075
HC          0.022075
AF          0.022075
HF          0.022075
AST         0.022075
HST         0.022075
AS          0.022075
HS          0.022075
Referee     0.022075
HTAG        0.022075
HTHG        0.022075
FTR         0.022075
FTAG        0.022075
FTHG        0.022075
AwayTeam    0.022075
Season      0.000000
dtype: float64


In [31]:
df = df.dropna()
print(len(df))
# Null analysis
print("\nNULL VALUES:")
print(df.isnull().sum().sort_values(ascending=False))

# Null percentage
print("\nNULL %:")
print((df.isnull().mean() * 100).sort_values(ascending=False))

4529

NULL VALUES:
Date        0
HomeTeam    0
AR          0
HR          0
AY          0
HY          0
AC          0
HC          0
AF          0
HF          0
AST         0
HST         0
AS          0
HS          0
Referee     0
HTAG        0
HTHG        0
FTR         0
FTAG        0
FTHG        0
AwayTeam    0
Season      0
dtype: int64

NULL %:
Date        0.0
HomeTeam    0.0
AR          0.0
HR          0.0
AY          0.0
HY          0.0
AC          0.0
HC          0.0
AF          0.0
HF          0.0
AST         0.0
HST         0.0
AS          0.0
HS          0.0
Referee     0.0
HTAG        0.0
HTHG        0.0
FTR         0.0
FTAG        0.0
FTHG        0.0
AwayTeam    0.0
Season      0.0
dtype: float64


In [32]:
df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,Referee,HS,...,AST,HF,AF,HC,AC,HY,AY,HR,AR,Season
0,2014-08-16,Arsenal,Crystal Palace,2.0,1.0,H,1.0,1.0,J Moss,14.0,...,2.0,13.0,19.0,9.0,3.0,2.0,2.0,0.0,1.0,2014/15
1,2014-08-16,Leicester,Everton,2.0,2.0,D,1.0,2.0,M Jones,11.0,...,3.0,16.0,10.0,3.0,6.0,1.0,1.0,0.0,0.0,2014/15
2,2014-08-16,Man United,Swansea,1.0,2.0,A,0.0,1.0,M Dean,14.0,...,4.0,14.0,20.0,4.0,0.0,2.0,4.0,0.0,0.0,2014/15
3,2014-08-16,QPR,Hull,0.0,1.0,A,0.0,0.0,C Pawson,19.0,...,4.0,10.0,10.0,8.0,9.0,1.0,2.0,0.0,0.0,2014/15
4,2014-08-16,Stoke,Aston Villa,0.0,1.0,A,0.0,0.0,A Taylor,12.0,...,2.0,14.0,9.0,2.0,8.0,0.0,3.0,0.0,0.0,2014/15
